In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("Services_Weekly.csv")

In [4]:
df.head()

,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,emergency,32,76,32,44,67,70,none
1,1,1,surgery,45,130,45,85,83,78,flu
2,1,1,general_medicine,37,201,37,164,97,43,flu
3,1,1,ICU,22,31,22,9,84,91,flu
4,2,1,emergency,28,169,28,141,75,64,none


In [5]:
df["bed_utilization_rate"] = (df["patients_admitted"] / df["available_beds"]) * 100

In [6]:
df[["service", "available_beds", "patients_admitted", "bed_utilization_rate"]].head(10)

,service,available_beds,patients_admitted,bed_utilization_rate
0,emergency,32,32,100.00
1,surgery,45,45,100.00
2,general_medicine,37,37,100.00
3,ICU,22,22,100.00
4,emergency,28,28,100.00
5,surgery,40,26,65.00
6,general_medicine,43,43,100.00
7,ICU,16,7,43.75
8,emergency,32,32,100.00
9,surgery,27,27,100.00


In [7]:
overall_bed_utilization = (
    df["patients_admitted"].sum() / df["available_beds"].sum()
) * 100

overall_bed_utilization

np.float64(92.69645120405576)

In [8]:
department_bed_utilization = (
    df.groupby("service")[["available_beds", "patients_admitted"]]
    .sum()
)

department_bed_utilization["bed_utilization_rate"] = (
    department_bed_utilization["patients_admitted"]
    / department_bed_utilization["available_beds"]
) * 100

department_bed_utilization

,available_beds,patients_admitted,bed_utilization_rate
service,,,
ICU,772,648,83.937824
emergency,1185,1185,100.000000
general_medicine,2404,2332,97.004992
surgery,1951,1686,86.417222


In [9]:
staff_df = pd.read_csv("Staff.csv")

staff_df.shape

(110, 4)

In [10]:
schedule_df = pd.read_csv("Staff_Schedule.csv")

schedule_df.shape

(6552, 6)

In [11]:
schedule_df["present"].value_counts()

present
1    3930
0    2622
Name: count, dtype: int64

In [12]:
staff_efficiency = (
    schedule_df.groupby("service")["present"]
    .agg(["sum", "count"])
)

staff_efficiency["staff_presence_rate"] = (
    staff_efficiency["sum"] / staff_efficiency["count"]
) * 100

staff_efficiency

,sum,count,staff_presence_rate
service,,,
ICU,1063,1768,60.124434
emergency,1225,2028,60.404339
general_medicine,859,1456,58.997253
surgery,783,1300,60.230769


In [14]:
efficiency_df = department_bed_utilization[
    ["bed_utilization_rate"]
].join(
    staff_efficiency[["staff_presence_rate"]]
)

efficiency_df

,bed_utilization_rate,staff_presence_rate
service,,
ICU,83.937824,60.124434
emergency,100.000000,60.404339
general_medicine,97.004992,58.997253
surgery,86.417222,60.230769


In [16]:
efficiency_df["department_efficiency_score"] = (
    efficiency_df["bed_utilization_rate"] * 0.5
    + efficiency_df["staff_presence_rate"] * 0.5
)

efficiency_df.round(2)

,bed_utilization_rate,staff_presence_rate,department_efficiency_score
service,,,
ICU,83.94,60.12,72.03
emergency,100.00,60.40,80.20
general_medicine,97.00,59.00,78.00
surgery,86.42,60.23,73.32
